# Snowflake: Managed Export to Ossie

Publishes `SALES_SV` to the shared Ossie file on S3 whenever the view changes. This side is
the source of truth and never imports, so downstream platforms are mirrors.

Runs two ways, and they call the same procedure:

- `CALL SYNC_OSSIE(...)` or `EXECUTE TASK`, for when you do not want to wait
- a task on a 1-minute schedule, for the background heartbeat

The only difference from the bidirectional notebook is the `ALLOWED` setting inside the
procedure. The decision function is identical, which is why this is a configuration choice
rather than a second implementation.

Runbook: `docs/DEMO_RUNBOOK_SNOWFLAKE_MANAGED.md`

## Step 1 - Set your database and schema

In [ ]:
DATABASE           = "DEMOS"
SCHEMA             = "SNOWFLAKE_MANAGED_SEMANTIC_INTEROP"
DATA_SCHEMA             = "EXT_SEMANTIC_INTEROP"
SEMANTIC_VIEW_NAME = "SALES_SV"
STAGE_NAME         = "SNOWFLAKE_MANAGED_OSSIE_STAGE"
MODEL_FILE         = "sales_model.yaml"          # shared, either side may write
STATE_FILE         = "_state/snowflake.json"     # this side only
WAREHOUSE          = "SI_DEMO_WH"
TASK_NAME          = "EXPORT_OSSIE_TASK"

print(f"Working in {DATABASE}.{SCHEMA}")

In [ ]:
USE ROLE ACCOUNTADMIN;
USE SCHEMA {{DATABASE}}.{{SCHEMA}};

## Step 2 - Verify the data and create the baseline semantic view

The Iceberg tables come from [`setup/snowflake_setup.sql`](../../../setup/snowflake_setup.sql).
If the next cell returns nothing, run that script first; see `setup/SETUP.md`.

The semantic view is then created in its **baseline state**: two metrics, one dimension pair,
one relationship. Everything the sync does afterwards starts from here, so running this cell
is also how you reset the Snowflake side between demo runs.

`CREATE OR REPLACE` is safe to re-run and is deliberately the simplest form of the view. The
extra metric arrives later, in step 8, as the change the sync has to carry.

In [ ]:
-- Both platforms read these same Parquet files from S3.
SELECT c.region,
       SUM(o.order_amount) AS total_order_amount,
       COUNT(o.order_id)   AS order_count,
       SUM(o.order_qty)    AS total_quantity
  FROM {{DATABASE}}.{{DATA_SCHEMA}}.ORDERS o
  JOIN {{DATABASE}}.{{DATA_SCHEMA}}.CUSTOMERS c USING (customer_id)
 GROUP BY c.region ORDER BY c.region;
-- expect EAST 750/5/12, WEST 700/5/11

In [ ]:
CREATE OR REPLACE SEMANTIC VIEW {{DATABASE}}.{{SCHEMA}}.{{SEMANTIC_VIEW_NAME}}
  TABLES (
    orders AS {{DATABASE}}.{{DATA_SCHEMA}}.ORDERS PRIMARY KEY (order_id),
    customers AS {{DATABASE}}.{{DATA_SCHEMA}}.CUSTOMERS PRIMARY KEY (customer_id)
  )
  RELATIONSHIPS (
    orders_to_customers AS orders (customer_id) REFERENCES customers (customer_id)
  )
  FACTS (
    orders.order_amount AS order_amount,
    orders.order_qty AS order_qty
  )
  DIMENSIONS (
    customers.region AS region,
    customers.customer_name AS customer_name
  )
  METRICS (
    orders.total_order_amount AS SUM(orders.order_amount),
    orders.order_count AS COUNT(orders.order_id)
  )
  COMMENT = 'Sales star for Ossie interop demo (Iceberg on S3)';

In [ ]:
SELECT * FROM SEMANTIC_VIEW(
  {{DATABASE}}.{{SCHEMA}}.{{SEMANTIC_VIEW_NAME}}
  DIMENSIONS customers.region
  METRICS orders.total_order_amount, orders.order_count
) ORDER BY region;
-- the baseline view: two metrics, and the numbers match the tables above

## Step 3 - The sync procedure

Python rather than SQL, because the fingerprint has to be byte-identical to the one running
on Databricks. Writing it twice, once in each language, is the most likely way to end up
with a sync that never converges.

The block between the `BEGIN GENERATED` markers comes from `assets/ossie_sync/` via
`assets/build_notebooks.py`. Edit the module and re-run the build rather than editing here.

In [ ]:
CREATE OR REPLACE PROCEDURE {{DATABASE}}.{{SCHEMA}}.SYNC_OSSIE(
    P_DATABASE      STRING,
    P_SCHEMA        STRING,
    P_SEMANTIC_VIEW STRING,
    P_STAGE         STRING,
    P_MODEL_FILE    STRING,
    P_STATE_FILE    STRING
)
RETURNS STRING
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python', 'pyyaml')
HANDLER = 'main'
EXECUTE AS CALLER
AS $$
# Snowflake side of the Ossie sync. Compares three fingerprints and acts once:
# export only; this side is the source of truth.
#
# Snowflake reads and writes Ossie natively, so this side needs neither the Apache
# converter nor the dialect shim. Only the fingerprint and the decision are shared
# with Databricks, and they are stamped in below from assets/ossie_sync/.
import hashlib
import io
import json
import re
from datetime import datetime, timezone

import yaml

# --- BEGIN GENERATED: ossie_sync.fingerprint ---
# Generated from assets/ossie_sync/fingerprint.py by assets/build_notebooks.py.
# Edit that file and re-run the build; changes made here are overwritten.
"""Canonical semantic fingerprint for an Ossie document.

Why this exists
---------------
The Snowflake <-> Databricks round trip is not byte-stable. The same semantic model
comes back with different dataset name casing, `primary_key` renamed to `unique_keys`,
facts dropped, dialect labels rewritten, and expressions gaining or losing table
qualifiers (`SUM(orders.order_amount)` on one side, `SUM(order_amount)` on the other).

So a sync that compares raw YAML, or a hash of it, never sees the two sides as equal and
writes forever. A sync that compares file timestamps is worse: every write makes the
writer the most recent change, so the model ping-pongs between platforms.

`semantic_fingerprint` solves this by hashing only the part of the model that both
platforms can express, in a normalized form that survives the trip. Two models with the
same fingerprint are treated as the same model, which is what lets the sync go quiet.

What is included
----------------
    tables          alias and source table, lowercased, last path component only
    relationships   from, to, and the join columns
    dimensions      qualified dimension names, sorted
    metrics         name and expression, sorted, table qualifiers stripped

What is excluded, and why
-------------------------
    Ossie `version`         differs by spec revision (0.1.1 against 0.2.0.dev0)
    dialect labels          SNOWFLAKE against ANSI_SQL against DATABRICKS
    comments, descriptions  Databricks does not round-trip them
    FACTS                   Snowflake-only concept, dropped by the converter
    relationship names      the converter rewrites their casing
    primary keys            Snowflake `primary_key` becomes `unique_keys` and back
    field and key order     not semantically meaningful

Excluding these has a real cost: editing only a comment, or only a fact, propagates
nothing. That is the deliberate trade. Including them would mean the two sides never
agree and the sync would write on every tick forever.
"""



FINGERPRINT_VERSION = "1"

# Matches a leading table qualifier on a column reference, e.g. the "orders." in
# "orders.order_amount". Stripped so that SUM(orders.order_amount) on the Snowflake side
# and SUM(order_amount) on the Databricks side produce the same fingerprint.
_QUALIFIER = re.compile(r"\b[A-Za-z_]\w*\.(?=[A-Za-z_]\w*)")
_WHITESPACE = re.compile(r"\s+")


def normalize_expression(expr):
    """Reduce a SQL expression to a comparable form.

    Lowercases, collapses whitespace, strips table qualifiers, and removes spaces
    around punctuation so that formatting differences do not register as changes.

        >>> normalize_expression("SUM( orders.order_amount )")
        'sum(order_amount)'
        >>> normalize_expression("sum(order_amount)")
        'sum(order_amount)'
    """
    if not expr:
        return ""
    text = _QUALIFIER.sub("", str(expr))
    text = _WHITESPACE.sub(" ", text).strip().lower()
    for token in ("(", ")", ",", "+", "-", "*", "/"):
        text = text.replace(" " + token, token).replace(token + " ", token)
    return text


def _last_identifier(source):
    """DEMOS.EXT_SEMANTIC_INTEROP.ORDERS -> orders"""
    return str(source or "").split(".")[-1].strip().strip('"').lower()


def _pick_expression(expression_obj):
    """Return the first expression string from an Ossie expression object.

    Dialect is ignored on purpose. The same expression labelled SNOWFLAKE, ANSI_SQL or
    DATABRICKS is the same expression for fingerprint purposes.
    """
    if not isinstance(expression_obj, dict):
        return ""
    for dialect in expression_obj.get("dialects") or []:
        if dialect.get("expression"):
            return dialect["expression"]
    return ""


def semantic_projection(ossie_yaml):
    """Reduce an Ossie document to the platform-neutral structure that gets hashed.

    Returned separately from the hash so notebooks can print it and show exactly what
    is being compared. When a sync will not converge, diffing two projections is the
    fastest way to find out which field is to blame.
    """
    root = yaml.safe_load(ossie_yaml) if isinstance(ossie_yaml, str) else ossie_yaml
    models = root.get("semantic_model") or []

    tables, relationships, dimensions, metrics = [], [], [], []

    for model in models:
        datasets = model.get("datasets") or []

        for dataset in datasets:
            alias = _last_identifier(dataset.get("name"))
            tables.append({"alias": alias, "source": _last_identifier(dataset.get("source"))})

            for field in dataset.get("fields") or []:
                # Only dimensions are portable. Snowflake facts have no `dimension` key
                # and are dropped by the Databricks converter, so including them here
                # would break convergence.
                if "dimension" not in field:
                    continue
                dimensions.append(f"{alias}.{str(field.get('name','')).lower()}")

        for rel in model.get("relationships") or []:
            # Ossie spells the join columns from_columns / to_columns. Only the join
            # columns are fingerprinted; the relationship's own name is not, because the
            # converter rewrites its casing (ORDERS_TO_CUSTOMERS -> ORDERS_to_CUSTOMERS).
            relationships.append({
                "from": _last_identifier(rel.get("from")),
                "to": _last_identifier(rel.get("to")),
                "from_columns": sorted(_last_identifier(c) for c in rel.get("from_columns") or []),
                "to_columns": sorted(_last_identifier(c) for c in rel.get("to_columns") or []),
            })

        # Snowflake stores metrics inside datasets[*].custom_extensions as a JSON blob;
        # the Apache converter uses a top-level `metrics` list. Read both.
        for metric in model.get("metrics") or []:
            metrics.append({
                "name": str(metric.get("name", "")).lower(),
                "expr": normalize_expression(_pick_expression(metric.get("expression"))),
            })

        for dataset in datasets:
            for ext in dataset.get("custom_extensions") or []:
                if ext.get("vendor_name") != "SNOWFLAKE":
                    continue
                try:
                    blob = json.loads(ext.get("data") or "{}")
                except (ValueError, TypeError):
                    continue
                for metric in blob.get("metrics") or []:
                    metrics.append({
                        "name": str(metric.get("name", "")).lower(),
                        "expr": normalize_expression(metric.get("expr")),
                    })

    def dedupe(rows, key):
        seen, out = set(), []
        for row in rows:
            marker = key(row)
            if marker not in seen:
                seen.add(marker)
                out.append(row)
        return out

    return {
        "fingerprint_version": FINGERPRINT_VERSION,
        "tables": sorted(dedupe(tables, lambda t: t["alias"]), key=lambda t: t["alias"]),
        "relationships": sorted(
            dedupe(relationships, lambda r: (r["from"], r["to"], tuple(r["from_columns"]))),
            key=lambda r: (r["from"], r["to"]),
        ),
        "dimensions": sorted(set(dimensions)),
        "metrics": sorted(dedupe(metrics, lambda m: m["name"]), key=lambda m: m["name"]),
    }


def semantic_fingerprint(ossie_yaml):
    """sha256 over the canonical projection. Stable across the round trip."""
    canonical = json.dumps(semantic_projection(ossie_yaml), sort_keys=True, separators=(",", ":"))
    return "sha256:" + hashlib.sha256(canonical.encode("utf-8")).hexdigest()


def describe(fingerprint):
    """Short form for log lines and notebook output."""
    if not fingerprint:
        return "(none)"
    return fingerprint.split(":")[-1][:12]
# --- END GENERATED: ossie_sync.fingerprint ---

# --- BEGIN GENERATED: ossie_sync.decide ---
# Generated from assets/ossie_sync/decide.py by assets/build_notebooks.py.
# Edit that file and re-run the build; changes made here are overwritten.
"""The sync decision: compare three fingerprints, return one verdict.

Both platforms and both architectures run this same function. The only thing that varies
is `allowed`, which is what stops the unidirectional variant from being a fork of the
bidirectional one.

The three inputs
----------------
    local   fingerprint of the model as it exists on this platform right now
    remote  fingerprint of the shared Ossie file on S3
    base    fingerprint this platform last agreed on, from its own state file

`base` is what makes this terminate. Without it there is no way to tell "the other side
changed" from "I changed", so both sides write and the model ping-pongs forever. With it,
each side can see which of the two moved, act once, record the new base, and go quiet.
"""

NO_CHANGE = "NO_CHANGE"
ADOPT = "ADOPT"
IMPORT = "IMPORT"
EXPORT = "EXPORT"
CONFLICT = "CONFLICT"
REVERT_LOCAL_DRIFT = "REVERT_LOCAL_DRIFT"

BIDIRECTIONAL = ("IMPORT", "EXPORT")
SNOWFLAKE_MANAGED_SOURCE = ("EXPORT",)      # Snowflake in the managed architecture
SNOWFLAKE_MANAGED_MIRROR = ("IMPORT",)      # Databricks in the managed architecture

REASONS = {
    NO_CHANGE: "local and shared model agree, nothing to do",
    ADOPT: "no recorded base, taking the shared model as the starting point",
    IMPORT: "shared model changed, replacing the local model",
    EXPORT: "local model changed, publishing to the shared Ossie file",
    CONFLICT: "both sides changed since the last agreement",
    REVERT_LOCAL_DRIFT: "local edit is not authoritative, restoring from the shared model",
}


class Decision:
    """A verdict plus the fingerprints that produced it, so it can be logged and read."""

    def __init__(self, action, reason, local, remote, base):
        self.action = action
        self.reason = reason
        self.local = local
        self.remote = remote
        self.base = base

    @property
    def writes(self):
        return self.action in (IMPORT, EXPORT, ADOPT, REVERT_LOCAL_DRIFT)

    def __str__(self):
        return f"{self.action} - {self.reason}"

    def to_dict(self):
        return {
            "action": self.action,
            "reason": self.reason,
            "local_fingerprint": self.local,
            "remote_fingerprint": self.remote,
            "base_fingerprint": self.base,
        }


def decide(local, remote, base, allowed=BIDIRECTIONAL, conflict_winner=None, platform=None):
    """Return a Decision.

    allowed
        Which write directions this platform may take. Bidirectional passes both.
        The managed architecture passes ("EXPORT",) on Snowflake and ("IMPORT",) on
        Databricks; an EXPORT that is not allowed becomes REVERT_LOCAL_DRIFT.

    conflict_winner, platform
        When both sides changed, the platform named by `conflict_winner` keeps its
        version. Anything else imports. Demoware: the losing edit is discarded with
        nothing more than a log line. See docs/PRODUCTION_ARCHITECTURE.md.
    """
    def verdict(action):
        return Decision(action, REASONS[action], local, remote, base)

    if local and remote and local == remote:
        return verdict(NO_CHANGE)

    if not local:
        # Nothing here yet, so there is no local change to protect.
        return verdict(ADOPT if remote else NO_CHANGE)

    if not remote:
        # Local model exists but the shared file does not.
        return verdict(EXPORT if EXPORT in allowed else NO_CHANGE)

    if base is None:
        return verdict(ADOPT)

    if local == base:
        action = IMPORT
    elif remote == base:
        action = EXPORT
    else:
        if conflict_winner and platform and conflict_winner == platform:
            return verdict(EXPORT if EXPORT in allowed else CONFLICT)
        if conflict_winner and platform:
            return verdict(IMPORT if IMPORT in allowed else CONFLICT)
        return verdict(CONFLICT)

    if action == EXPORT and EXPORT not in allowed:
        # Managed architecture: a locally edited mirror is drift, not a contribution.
        return verdict(REVERT_LOCAL_DRIFT)
    if action == IMPORT and IMPORT not in allowed:
        return verdict(NO_CHANGE)

    return verdict(action)


def next_base(decision):
    """The fingerprint to record after acting, or None to leave the base unchanged."""
    if decision.action in (IMPORT, ADOPT, REVERT_LOCAL_DRIFT):
        return decision.remote
    if decision.action == EXPORT:
        return decision.local
    return None
# --- END GENERATED: ossie_sync.decide ---

# --- BEGIN GENERATED: ossie_sync.state ---
# Generated from assets/ossie_sync/state.py by assets/build_notebooks.py.
# Edit that file and re-run the build; changes made here are overwritten.
"""Per-platform sync state, stored as JSON next to the shared Ossie file.

    s3://<bucket>/ossie/
      sales_model.yaml            shared model, either side may write
      _state/snowflake.json       written only by Snowflake
      _state/databricks.json      written only by Databricks

One writer per file, so there is no lock and no race. Each side reads only its own state
to answer "what did I last agree to", which is the `base` argument to decide().

Reading and writing the file is left to the caller, because the two runtimes do it very
differently: Databricks has dbutils.fs, Snowflake has stage COPY INTO. These helpers only
handle the JSON shape.
"""


STATE_VERSION = "1"


def new_state(base_fingerprint=None, last_action=None, platform=None):
    return {
        "state_version": STATE_VERSION,
        "base_fingerprint": base_fingerprint,
        "last_action": last_action,
        "by": platform,
        "at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }


def parse_state(text):
    """Tolerant read. A missing, empty or corrupt state file means no recorded base.

    Returning an empty state rather than raising is deliberate: decide() treats a base of
    None as ADOPT, which is the safe first-run behaviour and also the recovery path if
    someone deletes the file mid-demo.
    """
    if not text:
        return new_state()
    try:
        loaded = json.loads(text)
    except (ValueError, TypeError):
        return new_state()
    if not isinstance(loaded, dict):
        return new_state()
    return {
        "state_version": loaded.get("state_version", STATE_VERSION),
        "base_fingerprint": loaded.get("base_fingerprint"),
        "last_action": loaded.get("last_action"),
        "by": loaded.get("by"),
        "at": loaded.get("at"),
    }


def base_of(text):
    """The recorded base fingerprint from raw state-file text, or None."""
    return parse_state(text).get("base_fingerprint")


def serialize_state(state):
    return json.dumps(state, indent=2, sort_keys=True)


def state_after(decision, platform, next_base_fingerprint):
    """Build the state to persist after acting on a decision."""
    return new_state(
        base_fingerprint=next_base_fingerprint or decision.base,
        last_action=decision.action,
        platform=platform,
    )
# --- END GENERATED: ossie_sync.state ---

PLATFORM = "snowflake"
ALLOWED = SNOWFLAKE_MANAGED_SOURCE
CONFLICT_WINNER = "snowflake"   # demoware: see docs/PRODUCTION_ARCHITECTURE.md


def read_stage(session, stage_fqn, file_name, file_format):
    """Return a stage file's contents as text, or None if it is not there."""
    try:
        rows = session.sql(
            f"SELECT $1 FROM @{stage_fqn}/{file_name} (FILE_FORMAT => '{file_format}')"
        ).collect()
    except Exception:
        return None
    return rows[0][0] if rows and rows[0][0] else None


def write_stage(session, stage_fqn, file_name, body):
    session.file.put_stream(
        io.BytesIO(body.encode("utf-8")),
        f"@{stage_fqn}/{file_name}",
        auto_compress=False,
        overwrite=True,
    )


def local_ossie(session, view_fqn):
    """The Semantic View as Ossie YAML, or None if the view does not exist."""
    try:
        rows = session.sql(
            f"SELECT SYSTEM$READ_OSSIE_YAML_FROM_SEMANTIC_VIEW('{view_fqn}')"
        ).collect()
    except Exception:
        return None
    return rows[0][0] if rows else None


def main(session, p_database, p_schema, p_semantic_view, p_stage, p_model_file, p_state_file):
    namespace = f"{p_database}.{p_schema}"
    stage_fqn = f"{namespace}.{p_stage}"
    view_fqn = f"{namespace}.{p_semantic_view}"
    file_format = f"{namespace}.RAW_TEXT_FMT"

    # External stage directory metadata is cached; refresh before reading.
    session.sql(f"ALTER STAGE {stage_fqn} REFRESH").collect()

    shared_ossie = read_stage(session, stage_fqn, p_model_file, file_format)
    own_ossie = local_ossie(session, view_fqn)

    local = semantic_fingerprint(own_ossie) if own_ossie else None
    remote = semantic_fingerprint(shared_ossie) if shared_ossie else None
    base = base_of(read_stage(session, stage_fqn, p_state_file, file_format))

    decision = decide(local, remote, base, allowed=ALLOWED,
                      conflict_winner=CONFLICT_WINNER, platform=PLATFORM)

    if decision.action in (IMPORT, ADOPT, REVERT_LOCAL_DRIFT):
        # Snowflake imports Ossie natively. The model name inside the file decides the
        # view name, so keep it aligned with P_SEMANTIC_VIEW on the Databricks side.
        session.sql(
            "CALL SYSTEM$CREATE_SEMANTIC_VIEW_FROM_OSSIE_YAML(?, ?)",
            params=[namespace, shared_ossie],
        ).collect()

    elif decision.action == EXPORT:
        write_stage(session, stage_fqn, p_model_file, own_ossie)

    new_base = next_base(decision)
    if new_base:
        state = state_after(decision, PLATFORM, new_base)
        write_stage(session, stage_fqn, p_state_file, serialize_state(state))

    session.sql(f"ALTER STAGE {stage_fqn} REFRESH").collect()

    return (f"{decision.action} - {decision.reason} "
            f"[local={describe(local)} remote={describe(remote)} base={describe(base)}]")
$$;

## Step 4 - Run it by hand

Two equivalent ways in. `CALL` shows the procedure directly; `EXECUTE TASK` proves the
scheduled path works, and it runs even while the task is suspended. Use either during the
demo so nothing waits on the next tick.

In [ ]:
CALL {{DATABASE}}.{{SCHEMA}}.SYNC_OSSIE(
    '{{DATABASE}}', '{{SCHEMA}}', '{{SEMANTIC_VIEW_NAME}}',
    '{{STAGE_NAME}}', '{{MODEL_FILE}}', '{{STATE_FILE}}'
);

## Step 5 - What the semantic view holds now

In [ ]:
SHOW SEMANTIC METRICS IN {{DATABASE}}.{{SCHEMA}}.{{SEMANTIC_VIEW_NAME}};

SELECT $5 AS metric_name, $6 AS data_type
  FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));

In [ ]:
SELECT * FROM SEMANTIC_VIEW(
  {{DATABASE}}.{{SCHEMA}}.{{SEMANTIC_VIEW_NAME}}
  DIMENSIONS region
  METRICS total_order_amount, order_count
) ORDER BY region;
-- expect EAST 750/5, WEST 700/5

## Step 6 - The background task

Created suspended. Resume it for the live demo and suspend it immediately afterwards.

Offset from the Databricks job by roughly 30 seconds so the two are not writing at the same
instant. There is no lock: each side owns its own state file, and both write the shared
model only when something actually changed.

In [ ]:
CREATE OR REPLACE TASK {{DATABASE}}.{{SCHEMA}}.{{TASK_NAME}}
  WAREHOUSE = {{WAREHOUSE}}
  SCHEDULE = '1 MINUTE'
  COMMENT = 'Fingerprint-based Ossie sync. Suspended by default.'
AS
  CALL {{DATABASE}}.{{SCHEMA}}.SYNC_OSSIE(
      '{{DATABASE}}', '{{SCHEMA}}', '{{SEMANTIC_VIEW_NAME}}',
      '{{STAGE_NAME}}', '{{MODEL_FILE}}', '{{STATE_FILE}}'
  );

Run the task once without resuming it. This is the manual trigger to use mid-demo.

In [ ]:
EXECUTE TASK {{DATABASE}}.{{SCHEMA}}.{{TASK_NAME}};

In [ ]:
ALTER TASK {{DATABASE}}.{{SCHEMA}}.{{TASK_NAME}} RESUME;

In [ ]:
ALTER TASK {{DATABASE}}.{{SCHEMA}}.{{TASK_NAME}} SUSPEND;

## Step 7 - Watch the background runs

A column of `NO_CHANGE` with an occasional `IMPORT` or `EXPORT` is the point: the task runs every
minute and writes nothing until the model actually changes.

In [ ]:
SELECT scheduled_time, state, return_value, error_message
  FROM TABLE({{DATABASE}}.INFORMATION_SCHEMA.TASK_HISTORY(
      TASK_NAME => '{{TASK_NAME}}',
      SCHEDULED_TIME_RANGE_START => DATEADD('hour', -1, CURRENT_TIMESTAMP())))
 ORDER BY scheduled_time DESC
 LIMIT 20;

## Step 8 - Add a metric

Under this architecture every semantic change starts here. Add `TOTAL_QUANTITY` in
Snowflake, export, and it appears in every downstream mirror.

In [ ]:
CREATE OR REPLACE SEMANTIC VIEW {{DATABASE}}.{{SCHEMA}}.{{SEMANTIC_VIEW_NAME}}
  TABLES (
    orders AS {{DATABASE}}.{{DATA_SCHEMA}}.ORDERS PRIMARY KEY (order_id),
    customers AS {{DATABASE}}.{{DATA_SCHEMA}}.CUSTOMERS PRIMARY KEY (customer_id)
  )
  RELATIONSHIPS (
    orders_to_customers AS orders (customer_id) REFERENCES customers (customer_id)
  )
  FACTS (
    orders.order_amount AS order_amount,
    orders.order_qty AS order_qty
  )
  DIMENSIONS (
    customers.region AS region,
    customers.customer_name AS customer_name
  )
  METRICS (
    orders.total_order_amount AS SUM(orders.order_amount),
    orders.order_count AS COUNT(orders.order_id),
    orders.total_quantity AS SUM(orders.order_qty)
  )
  COMMENT = 'Sales star for Ossie interop demo (Iceberg on S3)';

In [ ]:
EXECUTE TASK {{DATABASE}}.{{SCHEMA}}.{{TASK_NAME}};   -- expect EXPORT

## Reset

Between demo runs, do both of these:

1. Re-run **step 2** above to put `SALES_SV` back to its baseline two metrics.
2. Run the cells below to clear this side's recorded base, so the next run starts from
   `ADOPT` rather than comparing against a fingerprint that no longer means anything.

Do the same on the Databricks side with `reset_demo()`.

A stale base fingerprint is the usual cause of an unexpected `CONFLICT` mid-demo.

In [ ]:
SELECT * FROM DIRECTORY(@{{DATABASE}}.{{SCHEMA}}.{{STAGE_NAME}})
 WHERE relative_path IN ('{{MODEL_FILE}}', '{{STATE_FILE}}');

In [ ]:
REMOVE @{{DATABASE}}.{{SCHEMA}}.{{STAGE_NAME}}/{{STATE_FILE}};
ALTER STAGE {{DATABASE}}.{{SCHEMA}}.{{STAGE_NAME}} REFRESH;